<a href="https://colab.research.google.com/github/bilalsherifdeen1-ux/AI-Project-Gallery/blob/main/AI-Project-Gallery/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install duckdb datasets scikit-learn --quiet

from google.colab import userdata
import duckdb

con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"""
CREATE SECRET hf_secret (
TYPE huggingface,
TOKEN '{userdata.get("HF_TOKEN")}'
);
""")

print("Setup complete.")

Setup complete.


In [ ]:
FEB_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet"
MARCH_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
CONTENT_PATH = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

def build_feature_frame(path):
    query = f"""
    WITH agg AS (
    SELECT
    client_hash_id, content_hash_id,
    SUM(gsc_clicks) AS clicks,
    SUM(gsc_impressions) AS impressions,
    SUM(gsc_avg_position * gsc_impressions) / NULLIF(SUM(gsc_impressions), 0) AS avg_position
    FROM read_parquet('{path}')
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) >= 100
    )
    SELECT
    a.client_hash_id, a.content_hash_id,
    a.avg_position, a.impressions,
    d.main_intent, d.search_volume, d.category_count,
    a.clicks * 1.0 / NULLIF(a.impressions, 0) AS actual_ctr
    FROM agg a
    JOIN read_parquet('{CONTENT_PATH}') d ON a.content_hash_id = d.content_hash_id
    """
    df = con.sql(query).df().dropna(subset=["actual_ctr", "avg_position", "main_intent",
                                             "search_volume", "category_count", "impressions"])
    numeric_cols = ["avg_position", "impressions", "search_volume", "category_count"]
    df[numeric_cols] = df[numeric_cols].astype("float64")
    return df

df_feb = build_feature_frame(FEB_PATH)
df_march = build_feature_frame(MARCH_PATH)
print("February rows:", df_feb.shape, " March rows:", df_march.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

February rows: (78396, 8)  March rows: (99197, 8)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error

numeric_features = ["avg_position", "impressions", "search_volume", "category_count"]
categorical_features = ["main_intent"]

def build_pipeline():
    pre = ColumnTransformer([
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ])
    return Pipeline([
        ("pre", pre),
        ("rf", RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)),
    ])

X_march = df_march[numeric_features + categorical_features]
y_march = df_march["actual_ctr"]
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X_march, y_march, test_size=0.25, random_state=42)
model_before = build_pipeline()
model_before.fit(X_train_b, y_train_b)
mae_before = mean_absolute_error(y_test_b, model_before.predict(X_test_b))

X_feb = df_feb[numeric_features + categorical_features]
y_feb = df_feb["actual_ctr"]
model_after = build_pipeline()
model_after.fit(X_feb, y_feb)
mae_after = mean_absolute_error(y_march, model_after.predict(X_march))

print(f"BEFORE (within-March, random split):   MAE = {mae_before:.5f}")
print(f"AFTER  (trained Feb, tested on March):  MAE = {mae_after:.5f}")
print(f"Gap: {mae_after - mae_before:.5f} ({(mae_after / mae_before - 1) * 100:+.1f}%)")

BEFORE (within-March, random split):   MAE = 0.00243
AFTER  (trained Feb, tested on March):  MAE = 0.00243
Gap: -0.00000 (-0.0%)
